# Treino do aluno (SFT + LoRA) na T4 do Colab

Fluxo: **dataset exportado pela API → treino → merge → GGUF → download → `ollama create` → `POST /treinos`**.

**Como usar (um run por sessão):**

1. `Ambiente de execução → Alterar tipo → GPU (T4)`.
2. Na célula **2. Condição e dataset**, escolha `CONDICAO` (`C2`, `C3` ou `C5`). É a única edição necessária.
3. `Ambiente de execução → Executar tudo`. Quando o seletor abrir, envie `treino.jsonl` e `held_out.jsonl` da condição escolhida.
4. Ao final, o navegador baixa `modelo-q4_k_m.gguf`, `Modelfile`, `run.json` e `resumo.txt`.

**Para o próximo run:** `Ambiente de execução → Desconectar e excluir ambiente de execução` e comece de novo. A etapa do GGUF troca o `torch` da sessão por uma versão sem GPU; uma sessão que já exportou não treina mais (a primeira célula recusa).

Tempo esperado (Qwen3-1.7B, ~3.000 exemplos, 2 épocas): cerca de 1 h de treino mais 15 min de GGUF. O held-out entra **só** no eval loss; o script recusa rodar se algum id estiver nos dois arquivos.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available() and "+cpu" not in torch.__version__, (
    "Sessão sem GPU utilizável (torch " + torch.__version__ + "). Selecione a GPU T4 ou, se esta sessão já exportou um GGUF, "
    "use 'Desconectar e excluir ambiente de execução' e comece de novo.")
print("torch", torch.__version__, "- GPU ok")

## 1. Código do projeto

Clona o repositório (ajuste `REPO_URL` se usar outro) e instala as dependências. Pode ser rodada de novo sem clonar em duplicidade.

In [ ]:
import os
REPO_URL = "https://github.com/ramonmserra/slm-privacy-distill.git"
%cd /content
if not os.path.exists("repo"):
    !git clone --depth 1 $REPO_URL repo
%cd /content/repo
!git pull -q
!pip install -q -r treino/requirements-treino.txt
# O Colab traz um torchao antigo que o peft recusa ao aplicar o LoRA; o treino nao usa torchao.
!pip uninstall -y -q torchao

## 2. Condição e dataset

Escolha `CONDICAO`. A célula define a versão do dataset, o modelo-base e o Modelfile, e confere se os arquivos enviados são os da condição (pela contagem de linhas).

| Condição | Modelo-base | Dataset a enviar |
|---|---|---|
| `C3` | Qwen3-1.7B | `v1-filtrado` |
| `C2` | Qwen3-1.7B | `v1-sem-filtro` |
| `C5` | Manacá-1B | `v1-filtrado` |

Se o dataset for reconstruído na API, a versão e as contagens mudam: atualize `CONDICOES` com os valores do novo `manifesto.json`.

In [ ]:
import glob
from google.colab import files

CONDICAO = "C3"  # C2 | C3 | C5  <- unica linha a editar

MANACA = '--modelo-base menezesbruno/manaca-1b-base --prefixo-do-assistente ""'
CONDICOES = {
    # versao e contagens vem do manifesto.json de cada dataset
    "C3": dict(versao="5b1301bbf406", treino=2592, held_out=454, extras="", modelfile="treino/Modelfile.template"),
    "C2": dict(versao="d6c7d57c179e", treino=3150, held_out=548, extras="", modelfile="treino/Modelfile.template"),
    "C5": dict(versao="5b1301bbf406", treino=2592, held_out=454, extras=MANACA, modelfile="treino/Modelfile.sem-chat.template"),
}
cfg = CONDICOES[CONDICAO]
DATASET_VERSAO, EXTRAS, MODELFILE = cfg["versao"], cfg["extras"], cfg["modelfile"]

# Apaga envios anteriores: se o arquivo ja' existe, o Colab salva o novo como
# "treino (1).jsonl" e o treino usaria o ANTIGO sem avisar.
for antigo in glob.glob("treino*.jsonl") + glob.glob("held_out*.jsonl"):
    os.remove(antigo)
print(f"Condicao {CONDICAO}: envie treino.jsonl e held_out.jsonl do dataset {DATASET_VERSAO}")
enviados = files.upload()
assert {"treino.jsonl", "held_out.jsonl"} <= set(enviados), "envie os dois arquivos, com os nomes treino.jsonl e held_out.jsonl"
linhas = {nome: sum(1 for l in open(nome, encoding="utf-8") if l.strip()) for nome in ("treino.jsonl", "held_out.jsonl")}
assert linhas == {"treino.jsonl": cfg["treino"], "held_out.jsonl": cfg["held_out"]}, (
    f"arquivos nao sao os da condicao {CONDICAO}: esperado treino={cfg['treino']} e held_out={cfg['held_out']}, recebido {linhas}")
HELD_OUT = "--held-out held_out.jsonl"
BASE = f"python -m treino.sft_lora --perfil colab-t4 --dataset treino.jsonl {HELD_OUT} --dataset-versao {DATASET_VERSAO} {EXTRAS}"
print("ok:", linhas, "|", BASE)

## 3. Dry-run e teste rápido

O dry-run valida os dados sem gastar GPU. O teste treina com 8 exemplos (poucos minutos): se passar, as bibliotecas da sessão são compatíveis com o script.

In [ ]:
!$BASE --saida saida --dry-run
!$BASE --max-exemplos 8 --max-eval 8 --sem-merge --saida saida-teste
import json
assert json.load(open("saida-teste/run.json", encoding="utf-8"))["status"] == "concluido", "o teste de 8 exemplos falhou; veja o erro acima antes do treino completo"

## 4. Treino

`--perfil colab-t4` = fp16, LoRA r=16, seq 1024, 2 épocas; o modelo-base vem da condição. Mantenha a aba aberta: se a sessão cair, o treino recomeça do zero. Para a condição "sem raciocínio", exporte o dataset com `incluir_raciocinio=false` na API; o script não muda.

In [ ]:
!$BASE --saida saida

In [ ]:
run = json.load(open("saida/run.json", encoding="utf-8"))
assert run["status"] == "concluido", "o treino nao concluiu; veja o erro acima"
print(json.dumps({k: run[k] for k in ("status", "modelo_base", "dataset_versao", "duracao_s", "metricas", "modelo_local_id", "hardware")}, indent=2, ensure_ascii=False))

## 5. GGUF quantizado (o que o Ollama carrega em CPU) e resumo

Esta etapa instala as dependências do `llama.cpp`, que trocam o `torch` da sessão. **Depois dela, esta sessão não treina mais.**

In [ ]:
!bash treino/exportar_gguf.sh saida/merged saida/gguf $MODELFILE
assert os.path.exists("saida/gguf/modelo-q4_k_m.gguf") and os.path.exists("saida/gguf/Modelfile"), (
    "a exportacao para GGUF falhou (veja o erro acima). O modelo treinado continua em saida/merged: "
    "corrija e rode SO' esta celula de novo; nao e' preciso treinar outra vez.")
!cp saida/run.json saida/gguf/run.json
!rm -f saida/gguf/modelo-f16.gguf
!python -m treino.resumo saida/gguf/run.json saida/gguf/resumo.txt
!cat saida/gguf/resumo.txt

## 6. Download

Baixa os quatro arquivos (autorize "baixar vários arquivos" se o navegador perguntar). Guarde-os numa pasta própria da condição. Na máquina do Ollama, na pasta com os arquivos:

```
ollama create <modelo_local_id sem o prefixo local:> -f Modelfile
```

Depois registre o run na API com o `run.json` como corpo:

```
curl -X POST -H "X-API-Key: SUA_CHAVE" -H "Content-Type: application/json" --data-binary @run.json http://localhost:8000/api/v1/treinos
curl -X POST -H "X-API-Key: SUA_CHAVE" http://localhost:8000/api/v1/treinos/<id>/verificar
```

In [ ]:
for nome in ("resumo.txt", "run.json", "Modelfile", "modelo-q4_k_m.gguf"):
    files.download(f"saida/gguf/{nome}")